# Day 18 — Cosine similarity ↔ attention  ·  Milestone 2 wrap 🎓

**The payoff you asked for on day 0:** *how are cosine similarity (RAG) and attention related?* Now you can prove it. Attention scores are **dot products**; cosine similarity is a **normalized** dot product. Normalize $Q,K$ and attention becomes "softmax over cosine similarities."

## 1. Review (~5 min)

Recall **day 7** (cosine similarity) and **day 17** (attention).

In [ ]:
import numpy as np
rng = np.random.default_rng(18)

**R1.** (day 7) cosine similarity of two vectors.

In [ ]:
def r_cosine_similarity(a, b):
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_r1(fn):
    a = rng.normal(size=5); b = rng.normal(size=5)
    assert np.allclose(fn(a, b), np.dot(a, b)/(np.linalg.norm(a)*np.linalg.norm(b)))
    print("✅ Review R1 passed")

check_r1(r_cosine_similarity)

**R2.** (day 17) scaled dot-product attention.

In [ ]:
def r_attention(Q, K, V):
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def _attn(Q, K, V):
    d = Q.shape[1]; S = Q @ K.T / np.sqrt(d)
    S = S - S.max(axis=1, keepdims=True); E = np.exp(S); W = E / E.sum(axis=1, keepdims=True)
    return W @ V

def check_r2(fn):
    Q = rng.normal(size=(2, 4)); K = rng.normal(size=(3, 4)); V = rng.normal(size=(3, 5))
    assert np.allclose(np.asarray(fn(Q, K, V)), _attn(Q, K, V))
    print("✅ Review R2 passed")

check_r2(r_attention)

## 2. Lecture note (~10 min): same core, different dressing

**The shared core.** Both quantities start from the **dot product** $q\cdot k$:
- **Cosine similarity** (RAG, day 7) divides it by both lengths: $\dfrac{q\cdot k}{\lVert q\rVert\lVert k\rVert}=\cos\theta\in[-1,1]$ — pure angle, one number per pair.
- **Attention score** (day 17) divides only by the constant $\sqrt{d}$ and then **softmaxes over the keys** and uses the weights to blend values.

So attention is *softmax-weighted aggregation using a dot-product similarity*, and cosine similarity is the *angle-only, single-pair* version of that same similarity.

**The bridge.** If you L2-**normalize** the rows of $Q$ and $K$ first, then every score $q\cdot k$ already *equals* $\cos\theta$. Attention then becomes literally **softmax over cosine similarities**:
$$\operatorname{CosineAttention}(Q,K,V) = \operatorname{softmax}\!\big(\hat Q\hat K^\top\big)\,V.$$

**Knobs & effect.** Standard attention keeps vector magnitudes (a long key can dominate, and scores can grow until softmax saturates); cosine attention bounds every score to $[-1,1]$, which is more stable. And the **retrieval** you built in M1 is the special case of one query with $V$ = the documents themselves: the key with the highest cosine *is* the top-1 retrieved item.

**In the wild.** "Cosine attention" is used in **Swin Transformer v2** (Liu et al. 2022) precisely for that training stability; and RAG retrieval is attention's $n_q=1$, argmax-instead-of-softmax cousin.

## 3. Exercises (~15 min)

### Exercise 1 — cosine score matrix
Given `Q` `(n_q, d)` and `K` `(n_k, d)`, return the `(n_q, n_k)` matrix of **cosine similarities** $\hat Q\hat K^\top$ (normalize rows first). Every entry lies in $[-1, 1]$.

In [ ]:
def cosine_scores(Q, K):
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_e1(fn):
    Q = rng.normal(size=(3, 4)); K = rng.normal(size=(5, 4))
    S = np.asarray(fn(Q, K)); assert S.shape == (3, 5)
    assert S.min() >= -1 - 1e-9 and S.max() <= 1 + 1e-9
    for i in range(3):
        for j in range(5):
            assert np.allclose(S[i, j], np.dot(Q[i], K[j])/(np.linalg.norm(Q[i])*np.linalg.norm(K[j])))
    print("✅ Exercise 1 passed")

check_e1(cosine_scores)

### Exercise 2 — attention on normalized Q,K *is* cosine attention (the connection)
Implement **cosine attention** = $\operatorname{softmax}(\hat Q\hat K^\top)\,V$. Confirm it equals plain scaled-dot-product attention run on the **already-normalized** `Q, K` (with `d` = the dimension, the $\sqrt{d}$ matches). This is the equivalence, made concrete.

In [ ]:
def cosine_attention(Q, K, V):
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def _sr(S):
    E = np.exp(S - S.max(axis=1, keepdims=True)); return E / E.sum(axis=1, keepdims=True)

def check_e2(fn):
    Q = rng.normal(size=(3, 4)); K = rng.normal(size=(5, 4)); V = rng.normal(size=(5, 6))
    Qn = Q / np.linalg.norm(Q, axis=1, keepdims=True)
    Kn = K / np.linalg.norm(K, axis=1, keepdims=True)
    expected = _sr(Qn @ Kn.T) @ V
    assert np.allclose(np.asarray(fn(Q, K, V)), expected)
    print("✅ Exercise 2 passed")

check_e2(cosine_attention)

### Exercise 3 — retrieval is attention's special case (tie back to M1)
For a single query, the **argmax of its cosine-attention weights** is the **top-1 retrieved document** (day 9). Return that index. The check plants an aligned key and expects it back — closing the loop from RAG to attention.

In [ ]:
def top1_attended(q, K):
    # q: (d,). Return the index of the key with the largest cosine-attention weight.
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_e3(fn):
    d = 4; q = rng.normal(size=d)
    K = rng.normal(size=(6, d)); K[4] = 3.0 * q  # key 4 most aligned (same direction)
    idx = fn(q, K)
    assert idx == 4
    # matches day-9 cosine top-1 retrieval
    Kn = K / np.linalg.norm(K, axis=1, keepdims=True); qn = q / np.linalg.norm(q)
    assert idx == int(np.argmax(Kn @ qn))
    print("✅ Exercise 3 passed")

check_e3(top1_attended)

## Solutions (collapsed — peek only if stuck)

`ref_`-prefixed answers. Running the next cell validates everything against the checks above.

In [ ]:
def ref_cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def ref_attention(Q, K, V):
    d = Q.shape[1]
    S = Q @ K.T / np.sqrt(d)
    E = np.exp(S - S.max(axis=1, keepdims=True)); W = E / E.sum(axis=1, keepdims=True)
    return W @ V

def _normrows(X):
    return X / np.linalg.norm(X, axis=1, keepdims=True)

def _softrows(S):
    E = np.exp(S - S.max(axis=1, keepdims=True)); return E / E.sum(axis=1, keepdims=True)

def ref_cosine_scores(Q, K):
    return _normrows(Q) @ _normrows(K).T

def ref_cosine_attention(Q, K, V):
    return _softrows(ref_cosine_scores(Q, K)) @ V

def ref_top1_attended(q, K):
    return int(np.argmax(_normrows(K) @ (q / np.linalg.norm(q))))

check_r1(ref_cosine_similarity)
check_r2(ref_attention)
check_e1(ref_cosine_scores)
check_e2(ref_cosine_attention)
check_e3(ref_top1_attended)
print("\nAll reference solutions pass. 🎉  Milestone 2 complete — you can read attention, and you see why it IS cosine similarity with a softmax on top!")